<a href="https://colab.research.google.com/github/victoria-banks/7.2/blob/main/72.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
with open("parser.py", "w") as f:
    f.write('''import re

class ParseNode:
    def __init__(self, symbol, children=None):
        self.symbol = symbol
        self.children = children if children is not None else []

    def to_dict(self):
        if not self.children:
            return self.symbol
        return {self.symbol: [child.to_dict() for child in self.children]}

parsing_table = {
    "E": {"id": ["T", "E'"], "(": ["T", "E'"], "-": ["T", "E'"]},
    "E'": {"+": ["+", "T", "E'"], "-": ["-", "T", "E'"], ")": [], "$": []},
    "T": {"id": ["F", "T'"], "(": ["F", "T'"], "-": ["F", "T'"]},
    "T'": {"*": ["*", "F", "T'"], "/": ["/", "F", "T'"], "+": [], "-": [], ")": [], "$": []},
    "F": {"id": ["U"], "(": ["U"], "-": ["U"]},
    "U": {"-": ["-", "U"], "(": ["(", "E", ")"], "id": ["id"]}
}

def tokenize(input_str):
    raw_tokens = re.findall(r'\\d+(?:\\.\\d+)?|id|[()+*/-]', input_str)
    tokens, values = [], []
    for token in raw_tokens:
        if token.replace('.', '', 1).isdigit():
            tokens.append("id")
            values.append(float(token) if '.' in token else int(token))
        elif token == "id":
            tokens.append("id")
            values.append(1)
        else:
            tokens.append(token)
            values.append(token)
    tokens.append("$")
    values.append("$")
    return tokens, values

def parse(tokens):
    root = ParseNode("E")
    stack = [("$", None), ("E", root)]
    index = 0
    current = tokens[index]

    while stack:
        top_symbol, node = stack.pop()
        if top_symbol == current:
            if node:
                node.children.append(ParseNode(current))
            index += 1
            if index < len(tokens):
                current = tokens[index]
        elif top_symbol in parsing_table:
            if current in parsing_table[top_symbol]:
                production = parsing_table[top_symbol][current]
                p_nodes = []
                for sym in production:
                    child = ParseNode(sym)
                    node.children.append(child)
                    p_nodes.append((sym, child))
                if not production:
                    node.children.append(ParseNode("ε"))
                for sym, p_node in reversed(p_nodes):
                    stack.append((sym, p_node))
            else:
                if current == ")":
                    print(f"Error at token index {index}: Unmatched closing parenthesis or missing operand.")
                elif current == "$":
                    print(f"Error at token index {index}: Unexpected end of input.")
                elif current in ["+", "-", "*", "/"]:
                    print(f"Error at token index {index}: Unexpected operator '{current}'.")
                else:
                    print(f"Error at token index {index}: Unexpected token '{current}'.")
                return False, None
        else:
            if top_symbol == ")":
                print(f"Error at token index {index}: Expected ')' but found '{current}'.")
            else:
                print(f"Error at token index {index}: Expected '{top_symbol}' but found '{current}'.")
            return False, None

    if index == len(tokens):
        return True, root
    else:
        print(f"Error at token index {index}: Extra input after valid expression.")
        return False, None
''')

with open("test_cases.txt", "w") as f:
    f.write('''5 + 5 * 2
10 - 4 / 2
-5 + 3
-(4 * 2)
(10 + 20) * 3
( id + id
id id
* 5 + 3
(5 + 2))''')

from google.colab import files
files.download("parser.py")
files.download("test_cases.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import re

class ParseNode:
    def __init__(self, symbol, children=None):
        self.symbol = symbol
        self.children = children if children is not None else []

    def to_dict(self):
        if not self.children:
            return self.symbol
        return {self.symbol: [child.to_dict() for child in self.children]}

parsing_table = {
    "E": {
        "id": ["T", "E'"],
        "(": ["T", "E'"],
        "-": ["T", "E'"]
    },
    "E'": {
        "+": ["+", "T", "E'"],
        "-": ["-", "T", "E'"],
        ")": [],
        "$": []
    },
    "T": {
        "id": ["F", "T'"],
        "(": ["F", "T'"],
        "-": ["F", "T'"]
    },
    "T'": {
        "*": ["*", "F", "T'"],
        "/": ["/", "F", "T'"],
        "+": [],
        "-": [],
        ")": [],
        "$": []
    },
    "F": {
        "id": ["U"],
        "(": ["U"],
        "-": ["U"]
    },
    "U": {
        "-": ["-", "U"],
        "(": ["(", "E", ")"],
        "id": ["id"]
    }
}

def tokenize(input_str):
    raw_tokens = re.findall(r'\d+(?:\.\d+)?|id|[()+*/-]', input_str)
    tokens = []
    values = []

    for token in raw_tokens:
        if token.replace('.', '', 1).isdigit():
            tokens.append("id")
            values.append(float(token) if '.' in token else int(token))
        elif token == "id":
            tokens.append("id")
            values.append(1)
        else:
            tokens.append(token)
            values.append(token)

    tokens.append("$")
    values.append("$")
    return tokens, values

def parse(tokens):
    root = ParseNode("E")
    stack = [("$", None), ("E", root)]
    index = 0
    current = tokens[index]

    while stack:
        top_symbol, node = stack.pop()

        if top_symbol == current:
            if node:
                node.children.append(ParseNode(current))
            index += 1
            if index < len(tokens):
                current = tokens[index]
        elif top_symbol in parsing_table:
            if current in parsing_table[top_symbol]:
                production = parsing_table[top_symbol][current]

                p_nodes = []
                for sym in production:
                    child = ParseNode(sym)
                    node.children.append(child)
                    p_nodes.append((sym, child))

                if not production:
                    node.children.append(ParseNode("ε"))

                for sym, p_node in reversed(p_nodes):
                    stack.append((sym, p_node))
            else:
                if current == ")":
                    print(f"Error at token index {index}: Unmatched closing parenthesis or missing operand.")
                elif current == "$":
                    print(f"Error at token index {index}: Unexpected end of input.")
                elif current in ["+", "-", "*", "/"]:
                    print(f"Error at token index {index}: Unexpected operator '{current}'.")
                else:
                    print(f"Error at token index {index}: Unexpected token '{current}'.")
                return False, None
        else:
            if top_symbol == ")":
                print(f"Error at token index {index}: Expected ')' but found '{current}'.")
            else:
                print(f"Error at token index {index}: Expected '{top_symbol}' but found '{current}'.")
            return False, None

    if index == len(tokens):
        return True, root
    else:
        print(f"Error at token index {index}: Extra input after valid expression.")
        return False, None

def evaluate_tree(node, val_iter):
    sym = node.symbol

    if sym == "E":
        left = evaluate_tree(node.children[0], val_iter)
        return evaluate_E_prime(node.children[1], left, val_iter)

    elif sym == "T":
        left = evaluate_tree(node.children[0], val_iter)
        return evaluate_T_prime(node.children[1], left, val_iter)

    elif sym == "F":
        return evaluate_tree(node.children[0], val_iter)

    elif sym == "U":
        if len(node.children) == 1 and node.children[0].symbol == "id":
            return next(val_iter)
        elif node.children[0].symbol == "-":
            return -evaluate_tree(node.children[1], val_iter)
        elif node.children[0].symbol == "(":
            val = evaluate_tree(node.children[1], val_iter)
            return val

def evaluate_E_prime(node, left_val, val_iter):
    if node.children[0].symbol == "ε":
        return left_val
    op = node.children[0].symbol
    right_val = evaluate_tree(node.children[1], val_iter)
    res = (left_val + right_val) if op == "+" else (left_val - right_val)
    return evaluate_E_prime(node.children[2], res, val_iter)

def evaluate_T_prime(node, left_val, val_iter):
    if node.children[0].symbol == "ε":
        return left_val
    op = node.children[0].symbol
    right_val = evaluate_tree(node.children[1], val_iter)
    res = (left_val * right_val) if op == "*" else (left_val / right_val)
    return evaluate_T_prime(node.children[2], res, val_iter)

def run_pipeline(expr):
    print(f"\nEvaluating: '{expr}'")
    tokens, values = tokenize(expr)
    print(f"Tokens: {tokens}")

    success, tree = parse(tokens)
    if success:
        print("Parsing successful!")
        print("Parse Tree (Nested Dict):", tree.to_dict())

        num_values = [v for t, v in zip(tokens, values) if t == "id"]
        val_result = evaluate_tree(tree, iter(num_values))
        print(f"Evaluated Result: {val_result}")
    else:
        print("Parsing failed.")

test_expressions = [
    "5 + 5 * 2",
    "id - id / id",
    "- 5 + 10",
    "- ( 3 * 4 )",
    "( id + id",
    "id id",
]

for expr in test_expressions:
    run_pipeline(expr)


Evaluating: '5 + 5 * 2'
Tokens: ['id', '+', 'id', '*', 'id', '$']
Parsing successful!
Parse Tree (Nested Dict): {'E': [{'T': [{'F': [{'U': [{'id': ['id']}]}]}, {"T'": ['ε']}]}, {"E'": [{'+': ['+']}, {'T': [{'F': [{'U': [{'id': ['id']}]}]}, {"T'": [{'*': ['*']}, {'F': [{'U': [{'id': ['id']}]}]}, {"T'": ['ε']}]}]}, {"E'": ['ε']}]}]}
Evaluated Result: 15

Evaluating: 'id - id / id'
Tokens: ['id', '-', 'id', '/', 'id', '$']
Parsing successful!
Parse Tree (Nested Dict): {'E': [{'T': [{'F': [{'U': [{'id': ['id']}]}]}, {"T'": ['ε']}]}, {"E'": [{'-': ['-']}, {'T': [{'F': [{'U': [{'id': ['id']}]}]}, {"T'": [{'/': ['/']}, {'F': [{'U': [{'id': ['id']}]}]}, {"T'": ['ε']}]}]}, {"E'": ['ε']}]}]}
Evaluated Result: 0.0

Evaluating: '- 5 + 10'
Tokens: ['-', 'id', '+', 'id', '$']
Parsing successful!
Parse Tree (Nested Dict): {'E': [{'T': [{'F': [{'U': [{'-': ['-']}, {'U': [{'id': ['id']}]}]}]}, {"T'": ['ε']}]}, {"E'": [{'+': ['+']}, {'T': [{'F': [{'U': [{'id': ['id']}]}]}, {"T'": ['ε']}]}, {"E'": ['ε']